## Modules & functions import

In [13]:
import os
import sys
import pandas as pd
from pprint import pprint

In [2]:
sys.path.append("../common_key_functions/")

In [3]:
from constants_configs import MODELS_CONFIGS, IMAGENET_CONSTANTS
from invert_batch_imagenet_funcs import val_model_orig_recon, \
    get_grid_images_paths, split_comparison_val_image, \
    get_labels_for_val_from_batch, get_labels_for_val_from_names, \
    sort_labels_by_images

## Specific pipeline functions

In [ ]:
def val_inversion(networks_names: list[str],
                  orig_images_dirs: list[str],
                  recon_images_dirs: list[str],
                  labels_true_all: list[list | dict],
                  num_workers: int | None = 8,
                  top_k_preds: int | None = 1,
                  per_subset: bool | None = False,
                  get_top2_gap: bool | None = False,
                  print_details: bool | None = True) -> dict[ str:float ]:
    """
    Run inversion validation on original and reconstructed images for a list of networks.
    For each network, validates both the original images and the corresponding
    reconstructions, printing classification accuracy and optional top-k predictions.

    Args:
        networks_names:    List of network architecture names.
        orig_images_dirs:  List of directories containing original images (one per network).
        recon_images_dirs: List of directories containing reconstructed images (one per network).
        labels_true_all:   List of ground truth label lists (one per network). Contains lists if 
        num_workers:       Number of DataLoader workers.
        top_k_preds:       Number of top predictions to display when `print_details==True`.
        per_subset:        If `True`, validates on each class & batch size separately
        get_top2_gap:      If `True`, includes difference between top-2 predictions probabilities.
        print_details:     If `True`, prints per-image prediction details.
        
    Returns:
        Classification accuracy & number of correctly classified images for each network
    """
    networks_val_results = {}
    
    for network_name, orig_images_dir, recon_images_dir, labels_true in zip(networks_names, 
                                                                            orig_images_dirs, 
                                                                            recon_images_dirs, 
                                                                            labels_true_all): 
        print(f"\n\nNetwork: {network_name}")
        
        if isinstance(labels_true, dict):
            if per_subset:
                groups = {}
                for image_name, label in labels_true.items():   # image_name is considered to have format "bs_<batch>_<class>_<index>"
                    image_name_parts = image_name.split('_')
                    if image_name_parts[0] == "bs":             # groups = { (batch_size, class_id) : [images filenames] }
                        groups.setdefault((int(image_name_parts[1]), label), []).append(image_name)

                network_val_results = {}
                for (batch, class_id), image_names in sorted(groups.items()):
                    network_val_results[(batch, class_id)] = val_model_orig_recon(
                        network_name, 
                        [ os.path.join(orig_images_dir,  image_name  + ".png") for image_name in image_names ], 
                        [ os.path.join(recon_images_dir, image_name  + ".png") for image_name in image_names ], 
                        per_subset,
                        [ labels_true[image_name] for image_name in image_names ], 
                        class_info=f"batch_size={batch}, class={class_id} (n={len(image_names)})",
                        num_workers=num_workers,
                        top_k_preds=top_k_preds,
                        get_top2_gap=get_top2_gap,
                        print_details=print_details,
                    )
                    
                    networks_val_results[network_name] = network_val_results
            
            else:                            
                networks_val_results[network_name] = val_model_orig_recon(
                    network_name, 
                    orig_images_dir, 
                    recon_images_dir, 
                    per_subset,
                    sort_labels_by_images([labels_true], [orig_images_dir])[0],
                    num_workers=num_workers,
                    top_k_preds=top_k_preds,
                    get_top2_gap=get_top2_gap,
                    print_details=print_details,
                )
        else:
            networks_val_results[network_name] = val_model_orig_recon(
                    network_name, 
                    orig_images_dir, 
                    recon_images_dir, 
                    per_subset,
                    labels_true,
                    num_workers=num_workers,
                    top_k_preds=top_k_preds,
                    get_top2_gap=get_top2_gap,
                    print_details=print_details,
                )
        
    return networks_val_results

## Validation

### Constants setting

In [5]:
dataset_dir_imagenet = "/media/user/Hitachi/ILSVRC/Data/CLS-LOC"
dataset_name = "ImageNet"

classes_ids_test = "1, 10, 100, 999"
classes_ids_max_gap = "24, 79, 409, 701, 712, 850, 950, 953, 954"
classes_ids = classes_ids_test  + ", " + classes_ids_max_gap

classes_ids_int = [ int(class_id.strip()) for class_id in classes_ids.split(",") ]
classes_ids_n = len(classes_ids_int)

networks_names = ["regnet_x_3_2", "regnet_x_16", "resnet50", "vit_b_16"] # MODELS_CONFIGS.keys()
networks_batch_sizes = {
    "regnet_x_3_2" : [16, 8, 4, 2],
    "regnet_x_16" :  [4, 2],
    "resnet50":      [12, 8, 4, 2],
    "vit_b_16":      [4, 2]
}
networks_fake_batch_sizes = {
    network_name : classes_ids_n - 1
    for network_name in networks_names
}
select_best_n = 10
val_images_dir = "inversion_images_val_diff_batch_size_per_class"

### Data preparation from training

During training cycle the best samples of original images with their reconstructed versions are saved in a grid of the following format:

In [6]:
# for usual training results with name "best_orig_vs_recon_<classes_ids>.png"
# networks_grid_images_paths = get_grid_images_paths(
    # networks_names, 
    # val_images_dir, 
    # classes_ids_int
# )

# for renamed training results
networks_grid_images_paths = get_grid_images_paths(
    networks_names,
    val_images_dir,
    classes_ids_int,
    networks_fake_batch_sizes
)
pprint(networks_grid_images_paths, indent=4)

{   'regnet_x_16': [   'inversion_images_val_diff_batch_size_per_class/regnet_x_16/bs_4_954.png',
                       'inversion_images_val_diff_batch_size_per_class/regnet_x_16/bs_4_24.png',
                       'inversion_images_val_diff_batch_size_per_class/regnet_x_16/bs_4_79.png',
                       'inversion_images_val_diff_batch_size_per_class/regnet_x_16/bs_2_954.png',
                       'inversion_images_val_diff_batch_size_per_class/regnet_x_16/bs_2_24.png',
                       'inversion_images_val_diff_batch_size_per_class/regnet_x_16/bs_4_701.png',
                       'inversion_images_val_diff_batch_size_per_class/regnet_x_16/bs_2_701.png',
                       'inversion_images_val_diff_batch_size_per_class/regnet_x_16/bs_2_1.png',
                       'inversion_images_val_diff_batch_size_per_class/regnet_x_16/bs_2_100.png',
                       'inversion_images_val_diff_batch_size_per_class/regnet_x_16/bs_2_953.png',
                       'i

In [7]:
for network_name, grid_images_paths in networks_grid_images_paths.items():
    for grid_image_i, grid_image_path in enumerate(grid_images_paths):
        grid_image_batch_size = int(os.path.splitext(os.path.basename(grid_image_path))[0].split("_")[1])
        split_comparison_val_image(
            grid_image_path,
            os.path.dirname(grid_image_path),
            IMAGENET_CONSTANTS["size_center_crop"] + 2,     # empirically adjusted padding
            # MODELS_CONFIGS[network_name]["batch_size"],   # for usual validation
            batch_size=grid_image_batch_size,
            select_best_n=select_best_n
        )

In [7]:
# for normal validation
# true_labels = [
#     get_labels_for_val_from_batch(
#         classes_ids_int, 
#         max(MODELS_CONFIGS[network_name]["batch_size"], classes_ids_n), 
#         select_best_n
#     )
#     for network_name in networks_names
# ]

# for custon validation
true_labels = [ 
    {
        image_name : int(label)
        for image_name, label in get_labels_for_val_from_names(
            os.path.join(val_images_dir, network_name, "origs")
        ).items()
    }
    for network_name in networks_names 
]

### Inversion validation: all classes & batch sizes

In [7]:
networks_val_results = val_inversion(
    networks_names,
    [ os.path.join(val_images_dir, network_name, "origs")  for network_name in networks_names ],
    [ os.path.join(val_images_dir, network_name, "recons") for network_name in networks_names ],
    true_labels,
    top_k_preds=5,
    # per_subset=True
)



Network: regnet_x_3_2

Validating regnet_x_3_2...
    Original images validation:

File: bs_16_100_0000.png
    True class: 100
    Top-5 predictions: ['class 100 (0.86571687)', 'class 99 (0.00412089)', 'class 209 (0.00156408)', 'class 130 (0.00053112)', 'class 206 (0.00046248)']
File: bs_16_100_0001.png
    True class: 100
    Top-5 predictions: ['class 100 (0.80088049)', 'class 99 (0.00251125)', 'class 143 (0.00193558)', 'class 209 (0.00151382)', 'class 128 (0.00146557)']
File: bs_16_100_0002.png
    True class: 100
    Top-5 predictions: ['class 100 (0.79388237)', 'class 99 (0.00229704)', 'class 209 (0.00194525)', 'class 130 (0.00092607)', 'class 90 (0.00080192)']
File: bs_16_100_0003.png
    True class: 100
    Top-5 predictions: ['class 100 (0.84086388)', 'class 99 (0.00483343)', 'class 137 (0.00151535)', 'class 53 (0.00082991)', 'class 136 (0.00078841)']
File: bs_16_100_0004.png
    True class: 100
    Top-5 predictions: ['class 100 (0.75972027)', 'class 209 (0.00200340)', 'cla

In [8]:
pprint(networks_val_results, indent=4)

{   'regnet_x_16': {   'orig_accuracy': 100.0,
                       'orig_correct': 78,
                       'recon_accuracy': 53.84615384615385,
                       'recon_correct': 42,
                       'total_images': 78},
    'regnet_x_3_2': {   'orig_accuracy': 99.35897435897436,
                        'orig_correct': 310,
                        'recon_accuracy': 37.82051282051282,
                        'recon_correct': 118,
                        'total_images': 312},
    'resnet50': {   'orig_accuracy': 100.0,
                    'orig_correct': 312,
                    'recon_accuracy': 36.858974358974365,
                    'recon_correct': 115,
                    'total_images': 312},
    'vit_b_16': {   'orig_accuracy': 100.0,
                    'orig_correct': 78,
                    'recon_accuracy': 92.3076923076923,
                    'recon_correct': 72,
                    'total_images': 78}}


### Inversion validation: per each class & batch size

In [ ]:
networks_val_results_per_subset = val_inversion(
    networks_names,
    [ os.path.join(val_images_dir, network_name, "origs")  for network_name in networks_names ],
    [ os.path.join(val_images_dir, network_name, "recons") for network_name in networks_names ],
    true_labels,
    top_k_preds=2,
    per_subset=True,
    print_details=True
)



Network: regnet_x_3_2

Validating regnet_x_3_2 on subset: batch_size=2, class=1 (n=2)...

    Original images validation:
File: bs_2_1_0001.png
    True class: 1
    Top-5 predictions: ['class 1 (0.75445533)', 'class 425 (0.00128181)', 'class 449 (0.00088856)', 'class 448 (0.00083235)', 'class 660 (0.00080146)']
File: bs_2_1_0000.png
    True class: 1
    Top-5 predictions: ['class 1 (0.96462095)', 'class 58 (0.00055081)', 'class 130 (0.00038418)', 'class 0 (0.00035367)', 'class 393 (0.00021960)']
Classification accuracy: 2/2 (100.00%)

    Reconstructed images validation:
File: bs_2_1_0001.png
    True class: 1
    Top-5 predictions: ['class 1 (0.46619436)', 'class 29 (0.01805081)', 'class 794 (0.00406327)', 'class 393 (0.00247244)', 'class 115 (0.00203402)']
File: bs_2_1_0000.png
    True class: 1
    Top-5 predictions: ['class 996 (0.11060154)', 'class 991 (0.10413796)', 'class 1 (0.05311900)', 'class 110 (0.02933855)', 'class 108 (0.02791249)']
Classification accuracy: 1/2 (50.00

In [21]:
pprint(networks_val_results_per_subset, indent=4)

{   'regnet_x_16': {   (2, 1): {   'orig_accuracy': 100.0,
                                   'orig_correct': 2,
                                   'recon_accuracy': 100.0,
                                   'recon_correct': 2,
                                   'total_images': 2},
                       (2, 10): {   'orig_accuracy': 100.0,
                                    'orig_correct': 2,
                                    'recon_accuracy': 100.0,
                                    'recon_correct': 2,
                                    'total_images': 2},
                       (2, 24): {   'orig_accuracy': 100.0,
                                    'orig_correct': 2,
                                    'recon_accuracy': 100.0,
                                    'recon_correct': 2,
                                    'total_images': 2},
                       (2, 79): {   'orig_accuracy': 100.0,
                                    'orig_correct': 2,
                          

In [ ]:
networks_val_results_per_subset_dfs = {}

for network_name, results_bs_class in networks_val_results_per_subset.items():
    network_rows = []
    for (batch_size, class_id), metrics in results_bs_class.items():
        network_rows.append({
            "batch_size": batch_size,
            "class_id": class_id,
            "orig_accuracy": metrics["orig_accuracy"],
            "recon_accuracy": metrics["recon_accuracy"],            
        })
    networks_val_results_per_subset_dfs[network_name] = pd.DataFrame(network_rows)

networks_val_results_per_subset_dfs["regnet_x_3_2"]

KeyError: 'regnet_x_3_2'

: 